<a href="https://colab.research.google.com/github/eya-bouhmida/Multimodal-RAG-From-Scratch/blob/main/02_parsing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
# Cellule 2
import os, shutil
os.chdir('/content/Multimodal-RAG-From-Scratch')

if os.path.exists('data'):
    if os.path.islink('data'):
        os.unlink('data')
    else:
        shutil.rmtree('data')

os.symlink(
    '/content/drive/MyDrive/multimodal-rag-project/data',
    '/content/Multimodal-RAG-From-Scratch/data'
)

print(os.listdir('data/raw/'))
print(os.listdir('data/raw/'))

['pubmed_multimodal', 'has_fr', 'who_en']
['pubmed_multimodal', 'has_fr', 'who_en']


In [42]:
!pip install pymupdf pillow -q

In [43]:
import fitz  # PyMuPDF
import os
import json
from PIL import Image
import io

In [44]:
def parse_pdf(pdf_path):
    """
    Extrait le texte et les images de chaque page d'un PDF
    """
    try:
        doc = fitz.open(pdf_path)
        pages_data = []

        for page_num in range(len(doc)):
            page = doc[page_num]

            # Extraire le texte
            text = page.get_text()

            # Extraire les images
            images = []
            image_list = page.get_images()

            for img_index, img in enumerate(image_list):
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                image_ext = base_image["ext"]
                image = Image.open(io.BytesIO(image_bytes))

                images.append({
                    "index": img_index,
                    "format": image_ext,
                    "width": image.width,
                    "height": image.height,
                    "size_bytes": len(image_bytes)
                })

            pages_data.append({
                "page_num": page_num + 1,
                "text": text.strip(),
                "num_images": len(images),
                "images": images
            })

        doc.close()
        return pages_data

    except Exception as e:
        print(f"Erreur: {e}")
        return []

In [50]:
# Cellule 3
# Redéfinir le chemin et le premier PDF
who_path = 'data/raw/who_en/'
first_pdf = os.listdir(who_path)[0]
print(f"Test sur: {first_pdf}")

# Tester l'extraction des images
images_output = 'data/processed/images/'
saved = extract_and_save_images(who_path + first_pdf, images_output)
print(f"✅ {len(saved)} images sauvegardées!")

Test sur: diabete_guidelineeng0.pdf
✅ 28 images sauvegardées!


In [52]:
def extract_and_save_images(pdf_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    saved_images = []
    pdf_name = os.path.basename(pdf_path).replace('.pdf', '')

    for page_num in range(len(doc)):
        page = doc[page_num]
        for img_index, img in enumerate(page.get_images()):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]
            image_filename = f"{pdf_name}_p{page_num+1}_img{img_index+1}.{image_ext}"
            image_path = os.path.join(output_dir, image_filename)
            with open(image_path, 'wb') as f:
                f.write(image_bytes)
            saved_images.append({
                "image_path": image_path,
                "page_num": page_num + 1,
                "img_index": img_index + 1
            })

    doc.close()
    return saved_images

In [53]:
# Cellule suivante — Parser tous les PDFs
import time

def parse_all_pdfs(raw_dir, images_output_dir):
    """
    Parse tous les PDFs de tous les sous-dossiers
    """
    all_documents = []
    sources = ['who_en', 'has_fr', 'pubmed_multimodal']

    for source in sources:
        source_path = os.path.join(raw_dir, source)
        pdfs = [f for f in os.listdir(source_path) if f.endswith('.pdf')]

        print(f"\n📁 Processing {source}: {len(pdfs)} PDFs")

        for i, pdf_file in enumerate(pdfs):
            pdf_path = os.path.join(source_path, pdf_file)

            try:
                # Extraire texte
                pages = parse_pdf(pdf_path)

                # Extraire images
                saved_images = extract_and_save_images(pdf_path, images_output_dir)

                all_documents.append({
                    "filename": pdf_file,
                    "source": source,
                    "path": pdf_path,
                    "num_pages": len(pages),
                    "num_images": len(saved_images),
                    "pages": pages
                })

                print(f"  ✅ [{i+1}/{len(pdfs)}] {pdf_file} — {len(pages)} pages, {len(saved_images)} images")

            except Exception as e:
                print(f"  ❌ [{i+1}/{len(pdfs)}] {pdf_file} — Erreur: {e}")

    return all_documents

# Lancer le parsing
all_docs = parse_all_pdfs('data/raw/', 'data/processed/images/')
print(f"\n🎉 Total: {len(all_docs)} documents parsés!")


📁 Processing who_en: 100 PDFs
  ✅ [1/100] diabete_guidelineeng0.pdf — 88 pages, 28 images
  ✅ [2/100] diabete_guidelineeng2.pdf — 29 pages, 0 images
  ✅ [3/100] diabeteeng6.pdf — 124 pages, 124 images
  ✅ [4/100] diabete_guidelineeng3.pdf — 106 pages, 1 images
  ✅ [5/100] diabeteeng4.pdf — 62 pages, 36 images
  ✅ [6/100] diabeteeng5.pdf — 64 pages, 49 images
  ✅ [7/100] diabeteeng7.pdf — 12 pages, 60 images
  ✅ [8/100] diabeteeng8.pdf — 26 pages, 86 images
  ✅ [9/100] diabeteseng9.pdf — 5 pages, 4 images
  ✅ [10/100] diabetes10.pdf — 40 pages, 8 images
  ✅ [11/100] hypertensioneng1.pdf — 61 pages, 2 images
  ✅ [12/100] hypertensioneng2.pdf — 43 pages, 12 images
  ✅ [13/100] hypertensioneng3.pdf — 5 pages, 33 images
  ✅ [14/100] hypertensioneng4.pdf — 20 pages, 56 images
  ✅ [15/100] hypertensioneng5.pdf — 1 pages, 21 images
  ✅ [16/100] hypertensioneng6.pdf — 9 pages, 61 images
  ✅ [17/100] hypertensioneng7.pdf — 9 pages, 4 images
  ✅ [18/100] hypertensioneng8.pdf — 9 pages, 4 images


In [54]:
images = os.listdir('data/processed/images/')
print(f"Nombre d'images sauvegardées: {len(images)}")
print(f"Exemples: {images[:5]}")

Nombre d'images sauvegardées: 35869
Exemples: ['diabete_guidelineeng0_p1_img1.png', 'diabete_guidelineeng0_p4_img1.jpeg', 'diabete_guidelineeng0_p10_img1.jpeg', 'diabete_guidelineeng0_p11_img1.jpeg', 'diabete_guidelineeng0_p17_img1.jpeg']


In [55]:
# Sauvegarder all_docs en JSON sur Drive
import json

output_json = 'data/processed/parsed_documents.json'

# On retire les données trop lourdes avant de sauvegarder
docs_to_save = []
for doc in all_docs:
    docs_to_save.append({
        "filename": doc["filename"],
        "source": doc["source"],
        "num_pages": doc["num_pages"],
        "num_images": doc["num_images"],
        "pages": [
            {
                "page_num": p["page_num"],
                "text": p["text"],
                "num_images": p["num_images"]
            }
            for p in doc["pages"]
        ]
    })

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(docs_to_save, f, ensure_ascii=False, indent=2)

print(f"✅ {len(docs_to_save)} documents sauvegardés dans {output_json}")

✅ 250 documents sauvegardés dans data/processed/parsed_documents.json


In [60]:
import os
os.chdir('/content/Multimodal-RAG-From-Scratch')

!git config --global user.email "eyabouhmida@gmail.com"
!git config --global user.name "eya-bouhmida"

# Sauvegarder le notebook depuis Colab vers GitHub
# File → Save a copy in GitHub → notebooks/02_parsing_pipeline.ipynb

In [61]:
!git add notebooks/02_parsing_pipeline.ipynb
!git commit -m "add parsing pipeline notebook"
!git push

fatal: pathspec 'notebooks/02_parsing_pipeline.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    data/processed/.gitkeep

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	Multimodal-RAG-From-Scratch/
	data

no changes added to commit (use "git add" and/or "git commit -a")
fatal: could not read Username for 'https://github.com': No such device or address
